# 05 · Optimisation Walkthrough

The end-to-end pipeline: train the ensemble, project the next 5 gameweeks, solve the multi-gameweek MILP. Inspect the solver status, the recommended XI, the captain choice, and the objective's implicit trade between free transfers and point hits.

In [5]:
from gaffer.providers.historical_csv import HistoricalCsvProvider
from gaffer.providers.fpl_api import LiveFplApiProvider
from gaffer.services.model_cache import train_or_load_ensembles
from gaffer.services.prediction_service import predict_projections
from gaffer.services.optimization_service import optimize_squad

fpl = LiveFplApiProvider()
hist = HistoricalCsvProvider()
point, quantile = train_or_load_ensembles(fpl, hist)
proj = predict_projections(fpl=fpl, historical=hist, point_model=point, quantile_model=quantile, horizon_gws=5)
proj.projections.head()

expected_points  lower_80  upper_80
player_id gameweek                                     
1         34.0             3.885320       1.0  6.047085
          35.0             3.885320       1.0  6.047085
          36.0             3.396086       1.0  6.421424
          37.0             3.918896       1.0  6.217916
          38.0             3.433065       1.0  6.225529

In [6]:
result = optimize_squad(
    projections=proj.projections,
    players=proj.players,
    start_gw=fpl.get_current_gw(),
    horizon=5,
)
print(f'Status: {result.solver_status} · net points: {result.total_expected_points:.1f}')
for plan in result.plans:
    print(f'GW{plan.gameweek}: XI={len(plan.xi_ids)}, captain={plan.captain_id}, transfers={len(plan.transfers_in)}')

Status: Optimal · net points: 339.9
GW34: XI=11, captain=449, transfers=0
GW35: XI=11, captain=449, transfers=0
GW36: XI=11, captain=430, transfers=0
GW37: XI=11, captain=449, transfers=0
GW38: XI=11, captain=430, transfers=0


In [7]:
# Inspect the first-GW XI in detail.
plan = result.plans[0]
xi = proj.players.loc[list(plan.xi_ids)].copy()
xi['xPts'] = proj.projections.xs(plan.gameweek, level='gameweek').loc[xi.index, 'expected_points']
xi.sort_values('xPts', ascending=False)

,name,team,position,price,team_short,xPts
player_id,,,,,,
449,Bruno Borges Fernandes,Man Utd,MID,10.3,MUN,8.590800
733,Senne Lammens,Man Utd,GKP,5.1,MUN,6.202579
5,Gabriel dos Santos Magalhães,Arsenal,DEF,7.1,ARS,6.093074
381,Mohamed Salah,Liverpool,MID,14.0,LIV,5.234465
329,Harry Wilson,Fulham,MID,6.0,FUL,4.719264
387,Dominik Szoboszlai,Liverpool,MID,7.1,LIV,4.441929
64,Ollie Watkins,Aston Villa,FWD,8.7,AVL,4.436058
6,William Saliba,Arsenal,DEF,6.1,ARS,4.341284
30,Kai Havertz,Arsenal,FWD,7.3,ARS,4.136414


That's the full pipeline. The Streamlit app (`streamlit run app/Home.py`) wraps these calls in an interactive UI.